# 给 bias-only / temperature-only 补 Student TS

**任务2：只补学生校准及旧集统一评价。** 六个组件 checkpoint 分别在原400条 `val_fit` 上拟合正温度；raw/combined的六个历史温度直接复用，并在同一校准集重放核验。最终交付四条件 × 三种子 × TS前后 × 两个评估集。

不训练、不选新epoch、不调用Jev/CLIP。推荐 **Colab CPU**。`T_s` 是学生后处理温度，与教师目标温度、训练蒸馏温度 τ=2 分开。

本 notebook 自带代码及已核对的输入指纹，上传这一个文件即可；只需修改第2节路径。真实校准输入由你补充，本交付未运行真实温度拟合，也未预填实验成绩。

## 1. 挂载 Google Drive

In [1]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('本地运行，无需挂载。')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. 填写输入路径

- `ABLATION_INPUT`：2026-10-28完整2×2消融模型包，含12个 `best.pt`；ZIP或解压目录。
- `EVALUATION_INPUT`：`Stage2_统一评价_协议与输入_20261026.zip`；也可填完整旧统一评价结果包或其中的 `inputs` 目录。必须含冻结特征、清单及bootstrap数组。
- **`CALIBRATION_INPUT`（待你填写）**：原Day2/学生校准缓存中的 `features/val_fit.npz`，也支持它的上级目录或包含它的ZIP。可以直接使用原Day2结果根目录；如果旧包精简掉了特征，请使用当时Drive上保留的缓存目录。

需原始NPZ哈希 `6027def1a024462f8d6694d24eda024bbbd4e0b15a2ba90bf3df23e5cb4e46bb`。如果找不到，请补回原缓存，不要换成400条val_select、200条补充评估数据或修改哈希。无需原图。

输出自动创建新目录；重置运行时后从头运行，保留旧结果。

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import uuid

ABLATION_INPUT = '/content/drive/MyDrive/CLIP_Jev_Ablation_2x2_20261028'
EVALUATION_INPUT = '/content/drive/MyDrive/Stage2_统一评价_协议与输入_20261026.zip'
CALIBRATION_INPUT = '/content/drive/MyDrive/val_fit.npz'
OUTPUT_PARENT = '/content/drive/MyDrive'
DEVICE = 'cpu'
OUTPUT_DIR = str(Path(OUTPUT_PARENT) / ('CLIP_Jev_Component_StudentTS_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:6]))

for name, value in [('ABLATION_INPUT',ABLATION_INPUT),('EVALUATION_INPUT',EVALUATION_INPUT),('CALIBRATION_INPUT',CALIBRATION_INPUT)]:
    if not value or not Path(value).exists():
        raise FileNotFoundError(f'请先填写正确的 {name}：{value!r}')
print('结果目录：', OUTPUT_DIR)

结果目录： /content/drive/MyDrive/CLIP_Jev_Component_StudentTS_20261030T025735Z_48932a


## 3. 检查依赖
只需 NumPy 和 PyTorch。Colab通常自带；如果提示缺失，按提示安装后重跑本格。

In [3]:
import importlib.util
missing = [m for m in ['numpy','torch'] if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError('新建单元格运行 %pip install ' + ' '.join(missing))
import numpy as np, torch
print('NumPy:',np.__version__,'PyTorch:',torch.__version__,'Device:',DEVICE)

NumPy: 2.1.3 PyTorch: 2.11.0+cpu Device: cpu


## 4. 加载内嵌实现与输入指纹
无需修改。不导入输入包中的Python代码；checkpoint以 `weights_only=True` 读取。

In [4]:
BASE_SOURCE = '"""Inference-only 2x2 evaluation. No source-package code is imported."""\nfrom pathlib import Path, PurePosixPath\nimport csv, io, json, hashlib, platform, zipfile, os, datetime\nimport numpy as np\n\nMETHODS=[\'KD_raw\',\'KD_bias\',\'KD_temperature\',\'KD_calibrated\']\nSEEDS=[20261010,20261011,20261012]\nMETRICS=[\'macro_f1\',\'accuracy\',\'nll\',\'ece10\',\'brier\',\'f1_E\',\'f1_N\',\'f1_C\']\nCONTRASTS={\n \'bias_at_T1\':[ -1,1,0,0], \'bias_at_Tfull\':[0,0,-1,1],\n \'temperature_at_b0\':[-1,0,1,0], \'temperature_at_bfull\':[0,-1,0,1],\n \'full_minus_raw\':[-1,0,0,1], \'interaction\':[1,-1,-1,1]}\ndef require(ok,msg):\n    if not ok:raise ValueError(msg)\ndef sha(b):return hashlib.sha256(b).hexdigest()\ndef digest_file(p):\n    h=hashlib.sha256()\n    with Path(p).open(\'rb\') as f:\n        for b in iter(lambda:f.read(4*1024*1024),b\'\'):h.update(b)\n    return h.hexdigest()\ndef save(p,obj):\n    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)\n    p.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding=\'utf-8\')\ndef table(p,records):\n    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)\n    require(bool(records),\'Empty output: \'+str(p))\n    with p.open(\'w\',encoding=\'utf-8-sig\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=list(records[0]));w.writeheader();w.writerows(records)\ndef csvbytes(b):return list(csv.DictReader(io.StringIO(b.decode(\'utf-8-sig\'))))\n\nclass Package:\n    """Read selected members without extracting or executing supplied programs."""\n    def __init__(self,path,marker):\n        self.path=Path(path).expanduser().resolve();require(self.path.exists(),\'输入不存在：\'+str(self.path))\n        self.z=zipfile.ZipFile(self.path) if self.path.is_file() else None\n        if self.z:\n            names=self.z.namelist();require(len(names)==len(set(names)),\'ZIP成员重复\')\n            candidates=[n[:-len(marker)] for n in names if n==marker or n.endswith(\'/\'+marker)]\n        else:\n            candidates=[str(q.parents[len(PurePosixPath(marker).parts)-1].relative_to(self.path)).replace(\'\\\\\',\'/\')+\'/\' for q in self.path.rglob(marker)]\n            candidates=[\'\' if x==\'./\' else x for x in candidates]\n        require(len(candidates)==1,\'未唯一定位 \'+marker+\'；请指定该包或其直接根目录。\')\n        self.prefix=candidates[0]\n    def read(self,name):\n        require(not PurePosixPath(name).is_absolute() and \'..\' not in PurePosixPath(name).parts and \'\\\\\' not in name,\'非法包内路径\')\n        if self.z:return self.z.read(self.prefix+name)\n        q=(self.path/self.prefix/name).resolve();require(q.is_relative_to(self.path),\'路径越界\');return q.read_bytes()\n    def j(self,name):return json.loads(self.read(name))\n    def npz(self,name):\n        with np.load(io.BytesIO(self.read(name)),allow_pickle=False) as a:return {k:a[k].copy() for k in a.files}\n\ndef metrics(z,y):\n    z=np.asarray(z,dtype=np.float64);y=np.asarray(y,dtype=np.int64)\n    require(z.shape==(len(y),3) and np.isfinite(z).all() and np.isin(y,[0,1,2]).all(),\'非法预测/标签\')\n    m=z.max(1);lse=m+np.log(np.exp(z-m[:,None]).sum(1));p=np.exp(z-lse[:,None]);pred=z.argmax(1)\n    cm=np.bincount(3*y+pred,minlength=9).reshape(3,3);f=2*cm.diagonal()/np.maximum(cm.sum(0)+cm.sum(1),1)\n    conf=p.max(1);hit=pred==y;bins=np.minimum((conf*10).astype(int),9)\n    nll=lse-z[np.arange(len(y)),y];brier=((p-np.eye(3)[y])**2).sum(1)\n    vals=dict(macro_f1=float(f.mean()),accuracy=float(hit.mean()),nll=float(nll.mean()),ece10=float(sum(abs(((hit-conf)*(bins==b)).mean()) for b in range(10))),brier=float(brier.mean()),f1_E=float(f[0]),f1_N=float(f[1]),f1_C=float(f[2]))\n    return vals,p,pred,nll,brier\n\ndef bootstrap_scores(z,y,draws,chunk=250):\n    vals,p,pred,nll,brier=metrics(z,y);n=len(y);conf=p.max(1);bins=np.minimum((conf*10).astype(int),9)\n    A=np.column_stack([np.eye(9)[y*3+pred],nll,brier]+[(pred==y).astype(float)*(bins==b)-conf*(bins==b) for b in range(10)])\n    blocks=[]\n    for start in range(0,len(draws),chunk):\n        ix=draws[start:start+chunk];offset=np.arange(len(ix))[:,None]*n\n        W=np.bincount((ix+offset).ravel(),minlength=len(ix)*n).reshape(len(ix),n)\n        blocks.append(W@A)\n    a=np.concatenate(blocks);cm=a[:,:9].reshape(-1,3,3)\n    f=2*np.diagonal(cm,axis1=1,axis2=2)/np.maximum(cm.sum(1)+cm.sum(2),1)\n    scores=dict(macro_f1=f.mean(1),accuracy=np.trace(cm,axis1=1,axis2=2)/n,nll=a[:,9]/n,brier=a[:,10]/n,ece10=abs(a[:,11:]).sum(1)/n,f1_E=f[:,0],f1_N=f[:,1],f1_C=f[:,2])\n    for ix in sorted(set([0,min(123,len(draws)-1),len(draws)-1])):\n        direct=metrics(z[draws[ix]],y[draws[ix]])[0]\n        require(all(abs(direct[k]-scores[k][ix])<1e-10 for k in METRICS),\'Bootstrap direct validation failed\')\n    return scores\n\ndef prepare(ablation_path,eval_path,out,anchors,device=\'cpu\'):\n    import torch\n    ab=Package(ablation_path,\'report/status.json\')\n    # marker has a subdirectory: prefix must include only the package root\n    ev=Package(eval_path,\'evaluation_protocol_LOCKED.json\')\n    out=Path(out).resolve()\n    require(not (out/\'completion.json\').exists(),\'此目录已完成。请保留已有结果，并为新运行设置新的 OUTPUT_DIR。\')\n    for source in [ab.path,ev.path]:require(out!=source and (not source.is_dir() or not out.is_relative_to(source)),\'输出目录须独立于输入\')\n    require(sha(ab.read(\'protocol.json\'))==anchors[\'ablation_protocol_sha256\'],\'2×2协议不是已核对的版本\')\n    protocol=ab.j(\'protocol.json\');require(ab.j(\'report/status.json\')[\'complete\'],\'2×2未全部完成\')\n    require(sha(ev.read(\'evaluation_protocol_LOCKED.json\'))==anchors[\'evaluation_protocol_sha256\'],\'评价协议版本不符\')\n    manifest_hashes=ev.j(\'PACKAGE_SHA256.json\')\n    require(sha(ev.read(\'PACKAGE_SHA256.json\'))==anchors[\'evaluation_manifest_sha256\'],\'评价输入清单版本不符\')\n    def verified(name):\n        b=ev.read(name);require(name in manifest_hashes and sha(b)==manifest_hashes[name],\'评价输入哈希不符：\'+name);return b\n    datasets={}\n    for spec in ev.j(\'evaluation_protocol_LOCKED.json\')[\'datasets\']:\n        split=spec[\'split\'];b=verified(spec[\'features_file\']);require(sha(b)==spec[\'npz_sha256\'],\'特征哈希不符\')\n        with np.load(io.BytesIO(b),allow_pickle=False) as a:d={k:a[k].copy() for k in [\'x\',\'y\',\'ids\',\'image_ids\']}\n        rows=[json.loads(s) for s in verified(spec[\'manifest_file\']).decode(\'utf-8-sig\').splitlines() if s.strip()]\n        require(sha(ev.read(spec[\'manifest_file\']))==spec[\'manifest_sha256\'],\'子清单哈希不符\')\n        require(len(rows)==len(d[\'y\'])==spec[\'rows\'] and len(set(map(str,d[\'image_ids\'])))==len(rows),\'样本数/图片唯一性不符\')\n        require(d[\'ids\'].tolist()==[r[\'sample_id\'] for r in rows] and d[\'y\'].tolist()==[r[\'label\'] for r in rows] and list(map(str,d[\'image_ids\']))==[str(r[\'image_id\']) for r in rows],\'ID/标签/图片顺序不符\')\n        x=d[\'x\'];require(x.dtype==np.float32 and x.shape==(len(rows),2048) and np.isfinite(x).all(),\'特征维度/精度错误\')\n        v,t=x[:,:512],x[:,512:1024]\n        require(np.allclose(np.linalg.norm(v,axis=1),1,atol=1e-4) and np.allclose(np.linalg.norm(t,axis=1),1,atol=1e-4),\'特征未归一化\')\n        require(np.allclose(x[:,1024:1536],abs(v-t),atol=1e-6) and np.allclose(x[:,1536:],v*t,atol=1e-6),\'特征组合错误\')\n        with np.load(io.BytesIO(verified(f\'bootstrap/{split}_draws.npz\')),allow_pickle=False) as a:\n            draws=a[\'indices\'].copy();ordered=list(map(str,a[\'image_ids\']))\n        require(draws.shape==(10000,len(rows)) and np.issubdtype(draws.dtype,np.integer) and draws.min()>=0 and draws.max()<len(rows),\'抽样数组错误\')\n        require(len(set(ordered))==len(rows) and set(ordered)==set(map(str,d[\'image_ids\'])),\'抽样图片ID错误\')\n        lookup={str(s):i for i,s in enumerate(d[\'image_ids\'])};d[\'order\']=np.array([lookup[s] for s in ordered]);d[\'draws\']=draws\n        datasets[split]=d\n    require(set(datasets)=={\'historical_test\',\'new_dev_holdout\'},\'评价集不完整\')\n    require(not set(map(str,datasets[\'historical_test\'][\'image_ids\']))&set(map(str,datasets[\'new_dev_holdout\'][\'image_ids\'])),\'评价集图片重叠\')\n    with np.load(io.BytesIO(verified(\'features/val_select_replay.npz\')),allow_pickle=False) as a:dev={k:a[k].copy() for k in a.files}\n    devrows=[json.loads(s) for s in verified(\'manifests/val_select.jsonl\').decode(\'utf-8-sig\').splitlines() if s.strip()]\n    require(dev[\'x\'].shape==(400,2048),\'重放特征错误\')\n    if \'ids\' in dev:require(dev[\'ids\'].tolist()==[r[\'sample_id\'] for r in devrows],\'重放顺序错误\')\n    models=[]\n    for method in METHODS:\n        group=f\'groups/{method}_alpha0.75\';gp=ab.j(group+\'/protocol.json\');gsha=sha(ab.read(group+\'/protocol.json\'))\n        require(gp[\'parent_sha256\']==anchors[\'ablation_protocol_sha256\'],\'模型父协议错误\')\n        expected=dict(b=protocol[\'teacher_params\'][\'b\'] if method in [\'KD_bias\',\'KD_calibrated\'] else 0.,T=protocol[\'teacher_params\'][\'T\'] if method in [\'KD_temperature\',\'KD_calibrated\'] else 1.)\n        require(gp[\'teacher_params\']==expected and gp[\'config\'][\'lr\']==.0003 and gp[\'config\'][\'alpha\']==.75 and gp[\'config\'][\'tau\']==2.,\'2×2配置不符\')\n        for seed in SEEDS:\n            mid=f\'{method}_{seed}\';folder=group+\'/runs/\'+mid;done=ab.j(folder+\'/done.json\');ref=anchors[\'models\'][mid]\n            require(done[\'identity\']==dict(protocol_sha256=gsha,method=method,seed=seed),\'模型身份错配\')\n            b=ab.read(folder+\'/best.pt\');require(sha(b)==done[\'checkpoint_sha256\']==ref[\'checkpoint_sha256\'],\'checkpoint哈希不符：\'+mid)\n            history=ab.read(folder+\'/history.json\');require(sha(history)==done[\'history_sha256\'],\'训练历史哈希不符\')\n            hist=json.loads(history);require([r[\'epoch\'] for r in hist]==list(range(1,31)),\'训练轮数不全\')\n            best=max(hist,key=lambda r:(r[\'validation\'][\'macro_f1\'],-r[\'validation\'][\'nll\'],-r[\'epoch\']))\n            require(best[\'epoch\']==done[\'best_epoch\']==ref[\'best_epoch\'],\'选择规则不符\')\n            preds=ab.read(folder+\'/predictions.csv\');require(sha(preds)==ref[\'predictions_sha256\'],\'开发预测版本错误\')\n            rr=csvbytes(preds);require([r[\'sample_id\'] for r in rr]==[r[\'sample_id\'] for r in devrows] and [int(r[\'label\']) for r in rr]==[r[\'label\'] for r in devrows],\'开发预测ID/标签不符\')\n            models.append(dict(method=method,seed=seed,model_id=mid,checkpoint_sha256=sha(b),best_epoch=done[\'best_epoch\'],payload=b,reference=np.array([[float(r[\'logit_\'+k]) for k in \'ENC\'] for r in rr])))\n    require(device in [\'cpu\',\'cuda\'] and (device==\'cpu\' or torch.cuda.is_available()),\'所选设备不可用\')\n    torch.set_num_threads(2);torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True\n    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False;torch.use_deterministic_algorithms(True)\n    env=dict(torch=str(torch.__version__),numpy=np.__version__,python=platform.python_version(),device=device,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0) if device==\'cuda\' else None,dtype=\'float32\',batch_size=256,tf32=False)\n    lock=dict(scope=\'Retrospective exploratory evaluation on previously opened sets; not independent confirmation\',methods=METHODS,seeds=SEEDS,models=[{k:v for k,v in m.items() if k not in [\'payload\',\'reference\']} for m in models],environment=env,input_anchors=anchors,contrasts=CONTRASTS,metrics=METRICS,bootstrap=\'Reuse 10000 fixed paired image draws; seeds fixed; no multiplicity adjustment\',student_TS=False,training=False,refit=False,replay_tolerance=1e-4)\n    out.mkdir(parents=True,exist_ok=True);lp=out/\'evaluation_LOCKED.json\'\n    if lp.exists():require(json.loads(lp.read_text())==lock,\'输出目录已有不同协议；请使用新的输出目录\')\n    else:save(lp,lock)\n    print(\'输入核验完成：12 个模型，1000 + 200 条；协议已锁定。\',flush=True)\n    return dict(ab=ab,ev=ev,out=out,models=models,data=datasets,dev=dev,device=device,lock=lock)\n\ndef predict(model,x,device):\n    import torch\n    with torch.inference_mode():return np.concatenate([model(torch.from_numpy(np.ascontiguousarray(x[i:i+256])).to(device)).cpu().numpy() for i in range(0,len(x),256)])\ndef load_head(m,device):\n    import torch\n    net=torch.nn.Sequential(torch.nn.Linear(2048,512),torch.nn.ReLU(),torch.nn.Dropout(.1),torch.nn.Linear(512,3))\n    net.load_state_dict(torch.load(io.BytesIO(m[\'payload\']),map_location=\'cpu\',weights_only=True),strict=True)\n    return net.to(device).eval()\ndef replay(ctx):\n    rows=[]\n    for m in ctx[\'models\']:\n        z=predict(load_head(m,ctx[\'device\']),ctx[\'dev\'][\'x\'],ctx[\'device\']);ref=m[\'reference\']\n        diff=float(np.max(abs(z-ref)));mismatch=int((z.argmax(1)!=ref.argmax(1)).sum())\n        require(diff<=1e-4 and mismatch==0,f"{m[\'model_id\']} 开发重放失败：{diff}, {mismatch}；不要放宽阈值。")\n        rows.append(dict(model_id=m[\'model_id\'],max_abs_difference=diff,class_mismatches=mismatch));print(\'Replay PASS\',m[\'model_id\'],flush=True)\n    table(ctx[\'out\']/\'development_replay.csv\',rows);ctx[\'replay_pass\']=True\n\ndef evaluate(ctx):\n    require(ctx.get(\'replay_pass\',False),\'必须先通过12个模型的开发重放\')\n    ctx[\'logits\']={};ctx[\'points\']={};records=[];summaries=[];cmrows=[]\n    for split,d in ctx[\'data\'].items():\n        predictions=[]\n        for m in ctx[\'models\']:\n            z=predict(load_head(m,ctx[\'device\']),d[\'x\'],ctx[\'device\']);vals,p,pred,_,_=metrics(z,d[\'y\'])\n            key=(split,m[\'method\'],m[\'seed\']);ctx[\'logits\'][key]=z;ctx[\'points\'][key]=vals\n            records.append(dict(split=split,method=m[\'method\'],seed=m[\'seed\'],variant=\'raw\',**vals))\n            folder=ctx[\'out\']/\'logits\'/split;folder.mkdir(parents=True,exist_ok=True)\n            np.savez_compressed(folder/(m[\'model_id\']+\'.npz\'),logits=z,ids=d[\'ids\'],image_ids=d[\'image_ids\'],y=d[\'y\'])\n            for i in range(len(z)):predictions.append(dict(model_id=m[\'model_id\'],sample_id=str(d[\'ids\'][i]),image_id=str(d[\'image_ids\'][i]),gold=int(d[\'y\'][i]),prediction=int(pred[i]),**{f\'logit_{c}\':float(z[i,j]) for j,c in enumerate(\'ENC\')},**{f\'p_{c}\':float(p[i,j]) for j,c in enumerate(\'ENC\')}))\n            cm=np.bincount(3*d[\'y\']+pred,minlength=9).reshape(3,3)\n            for i in range(3):\n                for j in range(3):cmrows.append(dict(split=split,method=m[\'method\'],seed=m[\'seed\'],gold=\'ENC\'[i],predicted=\'ENC\'[j],count=int(cm[i,j]),row_fraction=float(cm[i,j]/max(cm[i].sum(),1))))\n        table(ctx[\'out\']/split/\'predictions.csv\',predictions)\n        for method in METHODS:\n            rr=[r for r in records if r[\'split\']==split and r[\'method\']==method]\n            summaries.append(dict(split=split,method=method,n_seeds=3,**{k+s:float(fn([r[k] for r in rr])) for k in METRICS for s,fn in [(\'_mean\',np.mean),(\'_seed_sd\',lambda x:np.std(x,ddof=1))]}))\n        print(\'Evaluation complete:\',split,flush=True)\n    table(ctx[\'out\']/\'per_seed.csv\',records);table(ctx[\'out\']/\'summary.csv\',summaries);table(ctx[\'out\']/\'confusion_matrices.csv\',cmrows)\n    ctx[\'summary\']=summaries\n\ndef intervals(ctx):\n    rows=[];deltas=[]\n    for split,d in ctx[\'data\'].items():\n        order=d[\'order\'];y=d[\'y\'][order];scores={}\n        for m in ctx[\'models\']:\n            scores[m[\'method\'],m[\'seed\']]=bootstrap_scores(ctx[\'logits\'][split,m[\'method\'],m[\'seed\']][order],y,d[\'draws\'])\n            print(\'Bootstrap:\',split,m[\'model_id\'],flush=True)\n        for name,w in CONTRASTS.items():\n            for seed in SEEDS:\n                deltas.append(dict(split=split,comparison=name,seed=seed,**{k:sum(a*ctx[\'points\'][split,method,seed][k] for method,a in zip(METHODS,w)) for k in METRICS}))\n            for k in METRICS:\n                samples=np.mean([sum(a*scores[method,seed][k] for method,a in zip(METHODS,w)) for seed in SEEDS],axis=0)\n                values=[r[k] for r in deltas if r[\'split\']==split and r[\'comparison\']==name];lo,hi=np.quantile(samples,[.025,.975])\n                rows.append(dict(split=split,comparison=name,metric=k,delta=float(np.mean(values)),paired_seed_sd=float(np.std(values,ddof=1)),ci_low=float(lo),ci_high=float(hi),crosses_zero=bool(lo<=0<=hi),role=\'exploratory_unadjusted\',bootstrap_repeats=10000))\n    table(ctx[\'out\']/\'paired_intervals.csv\',rows);table(ctx[\'out\']/\'paired_seed_deltas.csv\',deltas);ctx[\'intervals\']=rows\n\ndef export(ctx):\n    out=ctx[\'out\'];rows=ctx[\'summary\'];ci=ctx[\'intervals\']\n    lines=[\'# 2×2统一评价结果\',\'\', \'四条件各三种子；固定已有checkpoint；未训练、未拟合温度、未做Student TS。\', \'\', \'历史1000与补充200均已有使用史。本次全部区间为探索性、未作多重比较校正；固定训练种子的图片bootstrap不消除选择偏差。\',\'\', \'| 集合 | 方法 | F1 % ± SD | NLL | ECE % | Brier |\',\'|---|---|---:|---:|---:|---:|\']\n    for r in rows:lines.append(f"| {r[\'split\']} | {r[\'method\']} | {100*r[\'macro_f1_mean\']:.3f} ± {100*r[\'macro_f1_seed_sd\']:.3f} | {r[\'nll_mean\']:.6f} | {100*r[\'ece10_mean\']:.3f} | {r[\'brier_mean\']:.6f} |")\n    lines+=[\'\',\'## 成分效应与交互\',\'\', \'| 集合 | 对比 | 指标 | 差值 [95%区间] |\',\'|---|---|---|---:|\']\n    for r in ci:\n        if r[\'metric\'] not in [\'macro_f1\',\'nll\',\'ece10\',\'brier\']:continue\n        s=100 if r[\'metric\'] in [\'macro_f1\',\'ece10\'] else 1\n        lines.append(f"| {r[\'split\']} | {r[\'comparison\']} | {r[\'metric\']} | {s*r[\'delta\']:+.5f} [{s*r[\'ci_low\']:+.5f}, {s*r[\'ci_high\']:+.5f}] |")\n    lines+=[\'\',\'F1/ECE差值展示为百分点。正差值利于F1，负差值利于NLL/ECE/Brier。交互=combined−bias−temperature+raw。区间跨0不证明等效；交互不跨0也不证明内部表征机制。不要将新模型结果替换旧主表而保留旧区间。\']\n    (out/\'RESULTS_结果解读.md\').write_text(\'\\n\'.join(lines),encoding=\'utf-8\')\n    # Self-contained tabular fragment, no additional package beyond booktabs.\n    tex=[\'\\\\begin{table*}[t]\',\'\\\\centering\\\\small\',\'\\\\begin{tabular}{llrrrr}\\\\toprule\',r\'Set & Target & F1 (\\%) & NLL & ECE (\\%) & Brier \\\\ \\midrule\']\n    for r in rows:\n        label=r[\'method\'].replace(\'_\',r\'\\_\');split=\'Historical\' if r[\'split\']==\'historical_test\' else \'Supplementary\'\n        tex.append(f"{split} & {label} & {100*r[\'macro_f1_mean\']:.3f} & {r[\'nll_mean\']:.6f} & {100*r[\'ece10_mean\']:.3f} & {r[\'brier_mean\']:.6f}"+r\' \\\\\')\n    tex += [r\'\\bottomrule\\end{tabular}\',r\'\\caption{Fixed 2$\\times$2 ablation on previously examined evaluation subsets. Three-seed means; no student TS. Full seed SDs and exploratory paired intervals accompany this table.}\',r\'\\end{table*}\']\n    (out/\'ablation_evaluation_table.tex\').write_text(\'\\n\'.join(tex),encoding=\'utf-8\')\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    fig,axes=plt.subplots(1,2,figsize=(10,4.3));names=list(CONTRASTS)\n    for ax,metric in zip(axes,[\'macro_f1\',\'nll\']):\n        for j,(split,color) in enumerate([(\'historical_test\',\'#0072B2\'),(\'new_dev_holdout\',\'#D55E00\')]):\n            rr=[next(r for r in ci if r[\'split\']==split and r[\'metric\']==metric and r[\'comparison\']==n) for n in names];scale=100 if metric==\'macro_f1\' else 1\n            for i,r in enumerate(rr):\n                yy=i+(j-.5)*.18;ax.plot([r[\'ci_low\']*scale,r[\'ci_high\']*scale],[yy,yy],color=color);ax.plot(r[\'delta\']*scale,yy,\'o\' if j==0 else \'s\',color=color,label=split if i==0 else None)\n        ax.set_yticks(range(len(names)),[\'Bias at T=1\',\'Bias at T=T*\',\'T at b=0\',\'T at b=b*\',\'Combined - raw\',\'Interaction\']);ax.invert_yaxis();ax.axvline(0,ls=\'--\',color=\'gray\');ax.set_xlabel(\'Delta F1 (pp)\' if metric==\'macro_f1\' else \'Delta NLL\');ax.grid(axis=\'x\',alpha=.2)\n    axes[0].legend(fontsize=8);fig.suptitle(\'Exploratory 95% paired image-bootstrap intervals; fixed seeds\');fig.tight_layout()\n    for ext in [\'pdf\',\'png\']:fig.savefig(out/(\'factorial_intervals.\'+ext),dpi=200,bbox_inches=\'tight\')\n    plt.close(fig)\n    audit=dict(status=\'COMPLETE\',models=12,splits=2,metric_rows=24,prediction_rows=14400,interval_rows=len(ci),bootstrap_direct_checks=True,development_replay=True,created_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),scope=ctx[\'lock\'][\'scope\'])\n    save(out/\'completion.json\',audit)\n    hashes={p.relative_to(out).as_posix():digest_file(p) for p in sorted(out.rglob(\'*\')) if p.is_file() and p.name!=\'PACKAGE_SHA256.json\'}\n    save(out/\'PACKAGE_SHA256.json\',hashes)\n    archive=out.with_name(out.name+\'_results.zip\')\n    with zipfile.ZipFile(archive,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(out.rglob(\'*\')):\n            if p.is_file():z.write(p,p.relative_to(out).as_posix())\n    print(\'完成。结果包：\',archive,flush=True);return archive\n'
TS_SOURCE = '"""Complete fixed 2x2 student TS, using only the original calibration subset."""\nfrom pathlib import Path\nimport io, json, zipfile, math\nimport numpy as np\nimport evaluate_2x2 as e\n\nNEW_METHODS = [\'KD_bias\', \'KD_temperature\']\nCLASS_METRICS = [\'macro_f1\', \'accuracy\', \'f1_E\', \'f1_N\', \'f1_C\']\n\ndef read_fit_input(path, expected_hash):\n    p = Path(path).expanduser()\n    e.require(p.exists(), \'请填写 CALIBRATION_INPUT：原400条 val_fit.npz、所在目录或包含它的ZIP。\')\n    if p.is_file() and p.suffix.lower() == \'.npz\':\n        data = p.read_bytes()\n    elif p.is_file() and zipfile.is_zipfile(p):\n        with zipfile.ZipFile(p) as z:\n            e.require(len(z.namelist()) == len(set(z.namelist())), \'ZIP有重复成员\')\n            names = [n for n in z.namelist() if n == \'val_fit.npz\' or n.endswith(\'/val_fit.npz\')]\n            matches = [z.read(n) for n in names if e.sha(z.read(n)) == expected_hash]\n            e.require(bool(matches), \'ZIP未找到与原校准缓存哈希一致的 val_fit.npz\')\n            data = matches[0]\n    else:\n        e.require(p.is_dir(), \'校准输入须为NPZ、目录或ZIP\')\n        names = list(p.rglob(\'val_fit.npz\'))\n        matches = [q for q in names if e.digest_file(q) == expected_hash]\n        e.require(bool(matches), \'目录未找到与原校准缓存哈希一致的 val_fit.npz\')\n        data = matches[0].read_bytes()\n    e.require(e.sha(data) == expected_hash, \'val_fit.npz 哈希不符；不得换成val_select或评估特征\')\n    with np.load(io.BytesIO(data), allow_pickle=False) as a:\n        d = {k: a[k].copy() for k in [\'x\', \'y\', \'ids\', \'image_ids\']}\n    return d\n\ndef attach_calibration(ctx, path, meta, old_temperatures):\n    d = read_fit_input(path, meta[\'npz_sha256\'])\n    manifest = ctx[\'ev\'].read(\'manifests/val_fit.jsonl\')\n    e.require(e.sha(manifest) == meta[\'manifest_sha256\'], \'校准清单哈希不符\')\n    rows = [json.loads(s) for s in manifest.decode(\'utf-8-sig\').splitlines() if s.strip()]\n    e.require(len(rows) == 400 and d[\'x\'].shape == (400, 2048), \'必须是原400条校准集\')\n    e.require(d[\'ids\'].tolist() == [r[\'sample_id\'] for r in rows] and\n              d[\'y\'].tolist() == [r[\'label\'] for r in rows] and\n              list(map(str, d[\'image_ids\'])) == [str(r[\'image_id\']) for r in rows], \'校准ID、标签或顺序不符\')\n    x = d[\'x\']; v, t = x[:, :512], x[:, 512:1024]\n    e.require(x.dtype == np.float32 and np.isfinite(x).all(), \'校准特征精度或数值错误\')\n    e.require(np.allclose(np.linalg.norm(v, axis=1), 1, atol=1e-4) and\n              np.allclose(np.linalg.norm(t, axis=1), 1, atol=1e-4) and\n              np.allclose(x[:, 1024:1536], abs(v-t), atol=1e-6) and\n              np.allclose(x[:, 1536:], v*t, atol=1e-6), \'校准特征定义不符\')\n    fit_ids = set(map(str, d[\'image_ids\']))\n    select_rows = [json.loads(s) for s in ctx[\'ev\'].read(\'manifests/val_select.jsonl\').decode(\'utf-8-sig\').splitlines() if s.strip()]\n    e.require(not fit_ids & {str(r[\'image_id\']) for r in select_rows}, \'校准与选模图片重叠\')\n    for split, data in ctx[\'data\'].items():\n        e.require(not fit_ids & set(map(str, data[\'image_ids\'])), \'校准与评估图片重叠：\' + split)\n    old = {m[\'id\']: m for m in old_temperatures[\'models\']}\n    for m in ctx[\'models\']:\n        if m[\'method\'] not in NEW_METHODS:\n            r = old[m[\'model_id\']]\n            e.require(r[\'checkpoint_sha256\'] == m[\'checkpoint_sha256\'] and r[\'best_epoch\'] == m[\'best_epoch\'], \'历史TS模型身份不符\')\n            e.require(np.isfinite(r[\'T\']) and r[\'T\'] > 0, \'历史温度非法\')\n    lock = dict(kind=\'component_student_TS_v1\', fit_split=\'val_fit\', fit_n=400,\n                fit_meta=meta, new_methods=NEW_METHODS, reused_temperatures=old_temperatures,\n                grid=dict(min=.05, max=20., points=401, extra=1., tie_break=\'closest to T=1\'),\n                formula=\'softmax(student_logits / T_s)\', selection=\'val_fit NLL grid minimum\',\n                model_anchors=ctx[\'lock\'][\'input_anchors\'], training=False,\n                comparisons=[\'four factorial contrasts, full-minus-raw, interaction: before and after TS\',\n                             \'within each method: TS-minus-before\'],\n                uncertainty=\'10000 fixed paired image draws; fixed seeds and fitted temperatures; exploratory, unadjusted\',\n                scope=\'Existing historical 1000 and supplementary 200 only; no new evaluation set\')\n    p = ctx[\'out\']/\'student_TS_protocol_LOCKED.json\'\n    if p.exists(): e.require(json.loads(p.read_text(encoding=\'utf-8\')) == lock, \'已有TS协议不同，请换输出目录\')\n    else: e.save(p, lock)\n    ctx.update(fit_data=d, ts_lock=lock, old_temperatures=old)\n    print(\'400条校准输入核验通过；拟合方案已锁定。\', flush=True)\n\ndef fit_temperature(z, y):\n    z = np.asarray(z, dtype=np.float64)\n    grid = np.unique(np.r_[1., np.geomspace(.05, 20., 401)])\n    values = [e.metrics(z/t, y)[0][\'nll\'] for t in grid]\n    i = min(range(len(grid)), key=lambda i: (values[i], abs(math.log(grid[i]))))\n    return dict(T=float(grid[i]), fit_nll_raw=e.metrics(z, y)[0][\'nll\'],\n                fit_nll_calibrated=float(values[i]), at_boundary=i in [0, len(grid)-1],\n                grid=grid.tolist(), nll_grid=values)\n\ndef fit_students(ctx):\n    e.require(ctx.get(\'replay_pass\', False) and \'fit_data\' in ctx, \'先通过开发重放并核验校准集\')\n    d = ctx[\'fit_data\']; records = []; folder = ctx[\'out\']/\'calibration_logits\'; folder.mkdir(exist_ok=True)\n    for m in ctx[\'models\']:\n        z = e.predict(e.load_head(m, ctx[\'device\']), d[\'x\'], ctx[\'device\']).astype(np.float64)\n        fit = fit_temperature(z, d[\'y\'])\n        if m[\'method\'] in NEW_METHODS:\n            source = \'new_fit\'\n        else:\n            old = ctx[\'old_temperatures\'][m[\'model_id\']]\n            e.require(abs(fit[\'T\']-old[\'T\']) < 1e-12, \'历史TS重放的网格最优温度不一致：\'+m[\'model_id\'])\n            e.require(abs(fit[\'fit_nll_raw\']-old[\'fit_nll_raw\']) <= 1e-5 and\n                      abs(fit[\'fit_nll_calibrated\']-old[\'fit_nll_calibrated\']) <= 1e-5, \'历史校准NLL重放不符\')\n            fit[\'T\'] = old[\'T\']; source = \'reused_locked_temperature\'\n        e.require(fit[\'fit_nll_calibrated\'] <= fit[\'fit_nll_raw\'] + 1e-12, \'拟合NLL劣于T=1\')\n        e.require(np.array_equal(z.argmax(1), (z/fit[\'T\']).argmax(1)), \'校准集TS改变类别\')\n        np.savez_compressed(folder/(m[\'model_id\']+\'.npz\'), logits=z, y=d[\'y\'], ids=d[\'ids\'], image_ids=d[\'image_ids\'])\n        records.append(dict(id=m[\'model_id\'], method=m[\'method\'], seed=m[\'seed\'],\n                            checkpoint_sha256=m[\'checkpoint_sha256\'], best_epoch=m[\'best_epoch\'], source=source, **fit))\n        print(m[\'model_id\'], source, \'T_s =\', fit[\'T\'], \'BOUNDARY\' if fit[\'at_boundary\'] else \'\', flush=True)\n    result = dict(protocol_sha256=e.digest_file(ctx[\'out\']/\'student_TS_protocol_LOCKED.json\'),\n                  fit_npz_sha256=ctx[\'ts_lock\'][\'fit_meta\'][\'npz_sha256\'], models=records,\n                  note=\'Fit NLL is an optimization diagnostic, not evaluation performance\')\n    p = ctx[\'out\']/\'student_temperature_LOCKED.json\'\n    if p.exists(): e.require(json.loads(p.read_text(encoding=\'utf-8\')) == result, \'已有温度锁不同，请使用新输出目录\')\n    else: e.save(p, result)\n    ctx[\'temperatures\'] = {r[\'id\']: r[\'T\'] for r in records}; ctx[\'calibration\'] = result\n\ndef evaluate_students(ctx):\n    e.require(\'temperatures\' in ctx, \'先锁定全部12个温度，再运行评估\')\n    e.evaluate(ctx)\n    records = []; summaries = []; checks = []; points = {}; logits = {}\n    for split, d in ctx[\'data\'].items():\n        for m in ctx[\'models\']:\n            key = (split, m[\'method\'], m[\'seed\']); raw = ctx[\'logits\'][key].astype(np.float64)\n            scaled = raw / ctx[\'temperatures\'][m[\'model_id\']]\n            a, b = e.metrics(raw, d[\'y\'])[0], e.metrics(scaled, d[\'y\'])[0]\n            e.require(np.array_equal(raw.argmax(1), scaled.argmax(1)) and all(a[k] == b[k] for k in CLASS_METRICS), \'TS改变分类结果\')\n            checks.append(dict(split=split, model_id=m[\'model_id\'], class_mismatches=0, macro_f1_delta=b[\'macro_f1\']-a[\'macro_f1\']))\n            for variant, z, vals in [(\'before_TS\', raw, a), (\'after_TS\', scaled, b)]:\n                vk = (split, m[\'method\'], m[\'seed\'], variant); points[vk] = vals; logits[vk] = z\n                records.append(dict(split=split, method=m[\'method\'], seed=m[\'seed\'], variant=variant, T=1. if variant==\'before_TS\' else ctx[\'temperatures\'][m[\'model_id\']], **vals))\n            np.savez_compressed(ctx[\'out\']/\'logits\'/split/(m[\'model_id\']+\'_TS.npz\'),\n                                logits=scaled, raw_logits=raw, probabilities=e.metrics(scaled, d[\'y\'])[1],\n                                predictions=scaled.argmax(1), T=ctx[\'temperatures\'][m[\'model_id\']], ids=d[\'ids\'], image_ids=d[\'image_ids\'], y=d[\'y\'])\n        for method in e.METHODS:\n            for variant in [\'before_TS\', \'after_TS\']:\n                rr = [r for r in records if r[\'split\']==split and r[\'method\']==method and r[\'variant\']==variant]\n                summaries.append(dict(split=split, method=method, variant=variant, n_seeds=3,\n                    **{k+s:float(fn([r[k] for r in rr])) for k in e.METRICS for s,fn in [(\'_mean\',np.mean),(\'_seed_sd\',lambda x:np.std(x,ddof=1))]}))\n    e.table(ctx[\'out\']/\'student_TS_per_seed.csv\', records)\n    e.table(ctx[\'out\']/\'student_TS_summary.csv\', summaries)\n    e.table(ctx[\'out\']/\'argmax_F1_invariance.csv\', checks)\n    ctx.update(ts_points=points, ts_logits=logits, ts_summary=summaries)\n\ndef paired_intervals(ctx):\n    rows = []; deltas = []\n    for split, d in ctx[\'data\'].items():\n        order = d[\'order\']; scores = {}; y = d[\'y\'][order]\n        for m in ctx[\'models\']:\n            for variant in [\'before_TS\', \'after_TS\']:\n                key = (split, m[\'method\'], m[\'seed\'], variant)\n                scores[m[\'method\'],m[\'seed\'],variant] = e.bootstrap_scores(ctx[\'ts_logits\'][key][order], y, d[\'draws\'])\n            print(\'Paired bootstrap:\', split, m[\'model_id\'], flush=True)\n        comparisons = []\n        for variant in [\'before_TS\', \'after_TS\']:\n            for name, weights in e.CONTRASTS.items():\n                comparisons.append((variant, name, [(method,variant,w) for method,w in zip(e.METHODS, weights) if w]))\n        for method in e.METHODS:\n            comparisons.append((\'TS_minus_before\', method, [(method,\'after_TS\',1),(method,\'before_TS\',-1)]))\n        for variant, name, terms in comparisons:\n            for seed in e.SEEDS:\n                deltas.append(dict(split=split, variant=variant, comparison=name, seed=seed,\n                    **{k:sum(w*ctx[\'ts_points\'][split,m,seed,v][k] for m,v,w in terms) for k in e.METRICS}))\n            for k in e.METRICS:\n                samples = np.mean([sum(w*scores[m,s,v][k] for m,v,w in terms) for s in e.SEEDS], axis=0)\n                vals = [sum(w*ctx[\'ts_points\'][split,m,s,v][k] for m,v,w in terms) for s in e.SEEDS]\n                lo, hi = np.quantile(samples, [.025,.975])\n                rows.append(dict(split=split, variant=variant, comparison=name, metric=k,\n                                 delta=float(np.mean(vals)), paired_seed_sd=float(np.std(vals,ddof=1)),\n                                 ci_low=float(lo), ci_high=float(hi), crosses_zero=bool(lo<=0<=hi),\n                                 bootstrap_repeats=len(d[\'draws\']), role=\'exploratory_unadjusted_fixed_temperatures\'))\n    e.table(ctx[\'out\']/\'student_TS_paired_intervals.csv\', rows)\n    e.table(ctx[\'out\']/\'student_TS_paired_seed_deltas.csv\', deltas)\n    ctx[\'ts_intervals\'] = rows\n\ndef export_results(ctx):\n    out = ctx[\'out\']\n    e.require(len(ctx[\'ts_summary\']) == 16 and len(ctx[\'ts_intervals\']) == 256, \'结果不完整\')\n    e.require(all(len(d[\'draws\']) == 10000 for d in ctx[\'data\'].values()), \'正式结果必须使用10000次固定抽样\')\n    lines = [\'# 四格KD：Student TS补全结果\', \'\',\n             \'六个组件温度在原400条val_fit上新拟合；raw与combined沿用已锁定温度并重放校验。没有重新训练或选checkpoint。\',\n             \'历史1000与补充200均为已查看的旧评估集。区间为探索性、未校正多重比较；条件于固定三个种子和拟合温度，未计入温度估计不确定性。\', \'\',\n             \'| 集合 | 方法 | 学生TS | F1 % ± SD | NLL ± SD | ECE % ± SD | Brier ± SD |\',\n             \'|---|---|---|---:|---:|---:|---:|\']\n    for r in ctx[\'ts_summary\']:\n        vals = [f"{r[k+\'_mean\']*s:.5f} ± {r[k+\'_seed_sd\']*s:.5f}" for k,s in [(\'macro_f1\',100),(\'nll\',1),(\'ece10\',100),(\'brier\',1)]]\n        lines.append(f"| {r[\'split\']} | {r[\'method\']} | {r[\'variant\']} | " + \' | \'.join(vals) + \' |\')\n    lines += [\'\', \'## 核心比较：双方完成Student TS后，temperature-only − raw\', \'\', \'| 集合 | 指标 | 差值 [95%配对区间] |\', \'|---|---|---:|\']\n    for r in ctx[\'ts_intervals\']:\n        if r[\'variant\']==\'after_TS\' and r[\'comparison\']==\'temperature_at_b0\' and r[\'metric\'] in [\'macro_f1\',\'nll\',\'ece10\',\'brier\']:\n            s=100 if r[\'metric\'] in [\'macro_f1\',\'ece10\'] else 1\n            lines.append(f"| {r[\'split\']} | {r[\'metric\']} | {r[\'delta\']*s:+.6f} [{r[\'ci_low\']*s:+.6f}, {r[\'ci_high\']*s:+.6f}] |")\n    lines += [\'\', \'F1/ECE差值为百分点。NLL、ECE、Brier负差值有利于前者。区间跨0不证明等效。\',\n              \'完整CSV包括：TS前/后各6种四格对比，以及每个方法自身TS前后差值；交互直接由联合抽样差分计算。\',\n              \'24个模型×集合组合的argmax与所有分类指标已检查完全不变。400条拟合NLL仅作优化诊断，不是独立评估成绩。\']\n    (out/\'RESULTS_结果解读.md\').write_text(\'\\n\'.join(lines),encoding=\'utf-8\')\n    e.save(out/\'completion.json\',dict(status=\'COMPLETE\',new_temperatures=6,reused_temperatures=6,\n                                    metric_rows=48,summary_rows=16,interval_rows=256,argmax_checks=24,training=False))\n    e.save(out/\'PACKAGE_SHA256.json\',{str(p.relative_to(out)).replace(\'\\\\\',\'/\'):e.digest_file(p)\n                                   for p in sorted(out.rglob(\'*\')) if p.is_file() and p.name!=\'PACKAGE_SHA256.json\'})\n    target = out.with_name(out.name+\'_results.zip\')\n    with zipfile.ZipFile(target,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(out.rglob(\'*\')):\n            if p.is_file():z.write(p,p.relative_to(out))\n    print(\'完成：\',target,flush=True)\n    return target\n'
INPUT_ANCHORS = {'ablation_protocol_sha256': '6079c58c8aeea255e401a6a9f5555f90523b038fe4f20e04d3d7c8319b050264', 'evaluation_protocol_sha256': 'f4e5acb3ca81cf8a2c27bd347b337c29cfaec9e6717c0793574ccd0a659ffe4e', 'evaluation_manifest_sha256': 'ac6915947711ffce7d18938e219df2ef4bb6f08370adc10ca0e1abf9374ed8d5', 'models': {'KD_raw_20261010': {'checkpoint_sha256': '4f2b53ea1887fd5d8630604fde0b07da46d2539f13f16adf5f9de6461053ec7b', 'best_epoch': 28, 'predictions_sha256': '64e754b9935ac7d61696018585c27c82dd2ac504cc318f54d2a708e3cb8fc4c2'}, 'KD_raw_20261011': {'checkpoint_sha256': '3d905cc39a34bab5a4a3021d82899be3d2c226e80a44a2060cad035f7d03e9fc', 'best_epoch': 21, 'predictions_sha256': 'fcef04ab8dbfd7cab802fefab493346894ac395e3566e58dc303d8272c3c280c'}, 'KD_raw_20261012': {'checkpoint_sha256': 'c1a157a362715deb96d1654cfead88c8d6e2c4506be34401210dfc19bfd016d5', 'best_epoch': 28, 'predictions_sha256': 'dcf1c4b7be143ac7556ff969e2cce454fa1ebe9196461a9ffcca861c3bf645ee'}, 'KD_bias_20261010': {'checkpoint_sha256': '995bdc60f942efdd1c15a76f42f80678f75cb265f0da38252232626c224a0391', 'best_epoch': 23, 'predictions_sha256': 'e2e4420721a1ed70912d0d390fb784a2af0e05bc1bc646db03856d857e93cd02'}, 'KD_bias_20261011': {'checkpoint_sha256': '17e7f127e231c513687dfdd60a512755750cc8929ac89dfc9c02de5304a6d1c8', 'best_epoch': 22, 'predictions_sha256': 'e6467a682ef31c05ff9568b6db3e4a54ad2fed5b081ac7ac565def06f23965be'}, 'KD_bias_20261012': {'checkpoint_sha256': '63cb1707523f4f4b194f8d9db81960da94a251dd7bbda6bc5bca93ac40c1283c', 'best_epoch': 21, 'predictions_sha256': '3628e8db7f3901a84e63c67da5fccc693a5c98b6eca1f9b2f5fc80f485a7bc65'}, 'KD_temperature_20261010': {'checkpoint_sha256': '54268dd7d79381e3ca302fcb516d7ee616deafdd3b178cf62d5c8e5c2f53c5b3', 'best_epoch': 23, 'predictions_sha256': '59143d5f02fbc963fa3ad31c012d533c996047c6712985127ee3bfb189d38f0d'}, 'KD_temperature_20261011': {'checkpoint_sha256': 'b3125ffe03be2918066b6a2611c8c4a765fd1d744c66e0216bc4bb5b49dadec0', 'best_epoch': 26, 'predictions_sha256': '26ec84b93c69de44b0ec9c5544d3546a12881e6a00031522ca39be256943421c'}, 'KD_temperature_20261012': {'checkpoint_sha256': '3013c5c9ba6f5f3dba89ee0019b33d5c570bfc4846c069f492ecb4cc2bbf22cd', 'best_epoch': 20, 'predictions_sha256': 'd730041dfb11e9a1dc36bd1a986aeb1155e3e515ccbb9dc043d6ef086367057d'}, 'KD_calibrated_20261010': {'checkpoint_sha256': 'c100fee94cf2d82c8e1656d4be468c1655c79a81817b678436c28e148c6a4a4f', 'best_epoch': 17, 'predictions_sha256': '01fa0f970700e36f7c2e4ed8a6ed6e9795b9d653a9b046cb7957f7af7e5da37f'}, 'KD_calibrated_20261011': {'checkpoint_sha256': 'c2a4ee53871bf0e222c045d8e0e9769573007ddc72a1a1ee698e9cd0f3010381', 'best_epoch': 21, 'predictions_sha256': '68fc38f23555c6bd86ec8a8ac3f5927f5861f1544634cd9cedc0a7678c30df21'}, 'KD_calibrated_20261012': {'checkpoint_sha256': '35b2d9d24c1b2926ce621fbfa428028cf7088e55dc39d7e2a2ec84338c5e5cec', 'best_epoch': 28, 'predictions_sha256': '3aafaff6a8e15e7d527203aed19deeb6c56a603bb3aed1f842d09eab05b2b79a'}}, 'evaluator_sha256': '05d52313ee54b2029a8855840f3a8636e35f9ce20bac60140e5f17ad97635270'}
FIT_META = {'npz_sha256': '6027def1a024462f8d6694d24eda024bbbd4e0b15a2ba90bf3df23e5cb4e46bb', 'manifest_sha256': 'a58e2f00047f159ca7b6415564ff03b699c68897fea805fbfb9088729fdbbaf4', 'revision': '3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268', 'frozen_encoder': True, 'feature_order': ['v', 't', 'abs(v-t)', 'v*t'], 'image_audit_sha256': '46dbe444ca7b9a02f0765be0b25e8700a969c3f2f1686e256bf8830febba6a87', 'compatibility_max_abs': 0.0, 'truncated_ids': [], 'extraction_environment': {'environment': {'torch': '2.11.0+cu130', 'numpy': '2.1.3', 'python': '3.13.15', 'device': 'cuda', 'cuda': '13.0', 'gpu': 'Tesla T4', 'cudnn': 92700, 'tf32': False, 'dtype': 'float32'}, 'transformers': '4.57.1', 'revision': '3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268'}}
OLD_TEMPERATURES = {'protocol_sha256': 'd16716e4ccb525bff863c51324aeca4f00700b8528564833a274a53b7442dde4', 'models': [{'id': 'CE_20261010', 'method': 'CE', 'seed': 20261010, 'checkpoint_sha256': '9aacd364c94b4bb5908aebc4fda71615d2b3202def2b51723a8caa3644e891ce', 'best_epoch': 23, 'T': 1.3903152918514001, 'at_boundary': False, 'fit_nll_raw': 0.8870456178107138, 'fit_nll_calibrated': 0.8660112768899446}, {'id': 'CE_20261011', 'method': 'CE', 'seed': 20261011, 'checkpoint_sha256': '912a21af6b911195463e18454a7db83da07c12f577ff34d420c0a6a9bc8f5f2a', 'best_epoch': 22, 'T': 1.3696454198636077, 'at_boundary': False, 'fit_nll_raw': 0.8856963902617824, 'fit_nll_calibrated': 0.8667695592524476}, {'id': 'CE_20261012', 'method': 'CE', 'seed': 20261012, 'checkpoint_sha256': '6576beb1d68c170e92925067f6a9fd834cf6edc3dd155e723bb91d7018b15d92', 'best_epoch': 21, 'T': 1.349282847673563, 'at_boundary': False, 'fit_nll_raw': 0.8834417428346302, 'fit_nll_calibrated': 0.8669149902660851}, {'id': 'KD_raw_20261010', 'method': 'KD_raw', 'seed': 20261010, 'checkpoint_sha256': '4f2b53ea1887fd5d8630604fde0b07da46d2539f13f16adf5f9de6461053ec7b', 'best_epoch': 28, 'T': 1.5210524830414918, 'at_boundary': False, 'fit_nll_raw': 0.8767818433627497, 'fit_nll_calibrated': 0.8390851627126478}, {'id': 'KD_raw_20261011', 'method': 'KD_raw', 'seed': 20261011, 'checkpoint_sha256': '3d905cc39a34bab5a4a3021d82899be3d2c226e80a44a2060cad035f7d03e9fc', 'best_epoch': 21, 'T': 1.4112971012222513, 'at_boundary': False, 'fit_nll_raw': 0.8648170754735607, 'fit_nll_calibrated': 0.8408611608333509}, {'id': 'KD_raw_20261012', 'method': 'KD_raw', 'seed': 20261012, 'checkpoint_sha256': 'c1a157a362715deb96d1654cfead88c8d6e2c4506be34401210dfc19bfd016d5', 'best_epoch': 28, 'T': 1.5210524830414918, 'at_boundary': False, 'fit_nll_raw': 0.8758655088854793, 'fit_nll_calibrated': 0.8387553647638984}, {'id': 'KD_calibrated_20261010', 'method': 'KD_calibrated', 'seed': 20261010, 'checkpoint_sha256': 'c100fee94cf2d82c8e1656d4be468c1655c79a81817b678436c28e148c6a4a4f', 'best_epoch': 17, 'T': 1.0940342035769433, 'at_boundary': False, 'fit_nll_raw': 0.8458297669664119, 'fit_nll_calibrated': 0.8443074497244336}, {'id': 'KD_calibrated_20261011', 'method': 'KD_calibrated', 'seed': 20261011, 'checkpoint_sha256': 'c2a4ee53871bf0e222c045d8e0e9769573007ddc72a1a1ee698e9cd0f3010381', 'best_epoch': 21, 'T': 1.1615863496415422, 'at_boundary': False, 'fit_nll_raw': 0.8466748624600743, 'fit_nll_calibrated': 0.8423801433625718}, {'id': 'KD_calibrated_20261012', 'method': 'KD_calibrated', 'seed': 20261012, 'checkpoint_sha256': '35b2d9d24c1b2926ce621fbfa428028cf7088e55dc39d7e2a2ec84338c5e5cec', 'best_epoch': 28, 'T': 1.2333095649679733, 'at_boundary': False, 'fit_nll_raw': 0.8473359665262457, 'fit_nll_calibrated': 0.8387973811671767}]}
TS_SOURCE_SHA256 = '66c9cb325d098512ec3bb66a4283a8f622c1de4252b0c85c909d85f388eb14ce'
import types, sys, hashlib
assert hashlib.sha256(BASE_SOURCE.encode()).hexdigest() == INPUT_ANCHORS['evaluator_sha256']
assert hashlib.sha256(TS_SOURCE.encode()).hexdigest() == TS_SOURCE_SHA256
e = types.ModuleType('evaluate_2x2')
sys.modules['evaluate_2x2'] = e
exec(compile(BASE_SOURCE,'<embedded_evaluator>','exec'),e.__dict__)
ts = types.ModuleType('student_ts')
sys.modules['student_ts'] = ts
exec(compile(TS_SOURCE,'<embedded_student_ts>','exec'),ts.__dict__)

## 5. 核验12个模型、全部输入并冻结拟合方案

检查checkpoint哈希、原选模规则、校准集ID和标签、特征定义、与选模/评估图片不重叠。保留旧评价核验记录 `evaluation_LOCKED.json`（描述原始logit阶段），新增 `student_TS_protocol_LOCKED.json` 才是本轮校准与分析协议。

温度网格沿用历史规则：`geomspace(0.05,20,401)` 加 `T=1`，最小化val_fit NLL；完全相等时优先接近1的温度。不依赖旧评估成绩挑温度。

In [5]:
ctx = e.prepare(ABLATION_INPUT,EVALUATION_INPUT,OUTPUT_DIR,INPUT_ANCHORS,device=DEVICE)
ts.attach_calibration(ctx,CALIBRATION_INPUT,FIT_META,OLD_TEMPERATURES)
(ctx['out']/'evaluate_2x2.py').write_text(BASE_SOURCE,encoding='utf-8')
(ctx['out']/'student_ts.py').write_text(TS_SOURCE,encoding='utf-8')
e.save(ctx['out']/'input_anchors.json',INPUT_ANCHORS)

输入核验完成：12 个模型，1000 + 200 条；协议已锁定。
400条校准输入核验通过；拟合方案已锁定。


## 6. 开发集重放
12个模型都须通过400条val_select的逐项重放：logit最大绝对差≤1e−4、类别完全一致。失败时停止核查，不放宽阈值。

In [6]:
e.replay(ctx)

Replay PASS KD_raw_20261010
Replay PASS KD_raw_20261011
Replay PASS KD_raw_20261012
Replay PASS KD_bias_20261010
Replay PASS KD_bias_20261011
Replay PASS KD_bias_20261012
Replay PASS KD_temperature_20261010
Replay PASS KD_temperature_20261011
Replay PASS KD_temperature_20261012
Replay PASS KD_calibrated_20261010
Replay PASS KD_calibrated_20261011
Replay PASS KD_calibrated_20261012


## 7. 拟合六个新温度，校验六个旧温度

保存每个模型的校准logits、完整网格NLL、checkpoint哈希和 `T_s`，得到 `student_temperature_LOCKED.json`。raw/combined重放必须得到同一网格温度，且拟合NLL差≤1e−5，否则停止。

如果温度位于搜索边界，会记录 `at_boundary=True`；不要根据评估成绩扩大网格。拟合NLL只是优化诊断，不是独立实验成绩。

In [7]:
ts.fit_students(ctx)
for r in ctx['calibration']['models']:
    print(r['id'],r['source'],'T_s=',r['T'],'boundary=',r['at_boundary'])

KD_raw_20261010 reused_locked_temperature T_s = 1.5210524830414918 
KD_raw_20261011 reused_locked_temperature T_s = 1.4112971012222513 
KD_raw_20261012 reused_locked_temperature T_s = 1.5210524830414918 
KD_bias_20261010 new_fit T_s = 1.4325955555491465 
KD_bias_20261011 new_fit T_s = 1.4112971012222513 
KD_bias_20261012 new_fit T_s = 1.3903152918514001 
KD_temperature_20261010 new_fit T_s = 1.1615863496415422 
KD_temperature_20261011 new_fit T_s = 1.2149739032812912 
KD_temperature_20261012 new_fit T_s = 1.144316999810911 
KD_calibrated_20261010 reused_locked_temperature T_s = 1.0940342035769433 
KD_calibrated_20261011 reused_locked_temperature T_s = 1.1615863496415422 
KD_calibrated_20261012 reused_locked_temperature T_s = 1.2333095649679733 
KD_raw_20261010 reused_locked_temperature T_s= 1.5210524830414918 boundary= False
KD_raw_20261011 reused_locked_temperature T_s= 1.4112971012222513 boundary= False
KD_raw_20261012 reused_locked_temperature T_s= 1.5210524830414918 boundary= False

## 8. 两个旧评估集的TS前后比较

48行逐种子指标，16行均值/样本SD：Macro-F1、Accuracy、各类F1、NLL、10-bin ECE、三分类Brier。
24个模型×集合组合都强制检查TS前后argmax、Macro-F1、Accuracy和各类F1完全不变；原始与缩放logits、概率、ID/标签均保存。

In [8]:
ts.evaluate_students(ctx)
for r in ctx['ts_summary']:
    print(f"{r['split']:18s} {r['method']:16s} {r['variant']:10s} F1={100*r['macro_f1_mean']:.3f}% NLL={r['nll_mean']:.6f} ECE={100*r['ece10_mean']:.3f}% Brier={r['brier_mean']:.6f}")

Evaluation complete: historical_test
Evaluation complete: new_dev_holdout
historical_test    KD_raw           before_TS  F1=60.729% NLL=0.908463 ECE=10.230% Brier=0.527304
historical_test    KD_raw           after_TS   F1=60.729% NLL=0.861393 ECE=3.438% Brier=0.509418
historical_test    KD_bias          before_TS  F1=60.631% NLL=0.907771 ECE=9.168% Brier=0.529415
historical_test    KD_bias          after_TS   F1=60.631% NLL=0.867702 ECE=3.412% Brier=0.513635
historical_test    KD_temperature   before_TS  F1=60.517% NLL=0.871301 ECE=4.990% Brier=0.513534
historical_test    KD_temperature   after_TS   F1=60.517% NLL=0.861616 ECE=3.049% Brier=0.509471
historical_test    KD_calibrated    before_TS  F1=59.866% NLL=0.870675 ECE=5.268% Brier=0.513471
historical_test    KD_calibrated    after_TS   F1=59.866% NLL=0.861816 ECE=3.160% Brier=0.509681
new_dev_holdout    KD_raw           before_TS  F1=63.500% NLL=0.870178 ECE=9.611% Brier=0.502395
new_dev_holdout    KD_raw           after_TS   F1=63

## 9. 配对bootstrap：所有方法使用同一套固定抽样

对两个集合分别使用原有10,000次图片抽样；先逐种子计算配对差，再平均三个种子，取2.5/97.5百分位。**不在bootstrap里重新拟合温度**，区间条件于固定checkpoint和温度。

| 对比 | 定义 |
|---|---|
| bias_at_T1 | bias − raw |
| bias_at_Tfull | combined − temperature |
| temperature_at_b0 | temperature − raw |
| temperature_at_bfull | combined − bias |
| full_minus_raw | combined − raw |
| interaction | combined − bias − temperature + raw |

以上六种对比在TS前、后分别计算；再计算四种方法各自的TS后−前。共2集合 × (6×2+4)对比 × 8指标 = **256行区间**。

重点看双方TS后的 `temperature_at_b0` 的NLL/Brier/ECE，以及 `bias_at_Tfull`。F1在TS前后不变，偏置所产生的分类差异也不会被正标量TS消除。旧集比较均为探索性、未作多重比较校正；区间跨0不能证明等效。

In [9]:
ts.paired_intervals(ctx)

Paired bootstrap: historical_test KD_raw_20261010
Paired bootstrap: historical_test KD_raw_20261011
Paired bootstrap: historical_test KD_raw_20261012
Paired bootstrap: historical_test KD_bias_20261010
Paired bootstrap: historical_test KD_bias_20261011
Paired bootstrap: historical_test KD_bias_20261012
Paired bootstrap: historical_test KD_temperature_20261010
Paired bootstrap: historical_test KD_temperature_20261011
Paired bootstrap: historical_test KD_temperature_20261012
Paired bootstrap: historical_test KD_calibrated_20261010
Paired bootstrap: historical_test KD_calibrated_20261011
Paired bootstrap: historical_test KD_calibrated_20261012
Paired bootstrap: new_dev_holdout KD_raw_20261010
Paired bootstrap: new_dev_holdout KD_raw_20261011
Paired bootstrap: new_dev_holdout KD_raw_20261012
Paired bootstrap: new_dev_holdout KD_bias_20261010
Paired bootstrap: new_dev_holdout KD_bias_20261011
Paired bootstrap: new_dev_holdout KD_bias_20261012
Paired bootstrap: new_dev_holdout KD_temperature_

## 10. 导出完整结果
生成结果报告、CSV、全部温度和预测、校验清单与ZIP。仅在全部检查通过后写入完成标记。

In [10]:
RESULT_ZIP = ts.export_results(ctx)
from IPython.display import display, Markdown
display(Markdown((ctx['out']/'RESULTS_结果解读.md').read_text(encoding='utf-8')))
print('结果已保存：',RESULT_ZIP)

完成： /content/drive/MyDrive/CLIP_Jev_Component_StudentTS_20261030T025735Z_48932a_results.zip


# 四格KD：Student TS补全结果

六个组件温度在原400条val_fit上新拟合；raw与combined沿用已锁定温度并重放校验。没有重新训练或选checkpoint。
历史1000与补充200均为已查看的旧评估集。区间为探索性、未校正多重比较；条件于固定三个种子和拟合温度，未计入温度估计不确定性。

| 集合 | 方法 | 学生TS | F1 % ± SD | NLL ± SD | ECE % ± SD | Brier ± SD |
|---|---|---|---:|---:|---:|---:|
| historical_test | KD_raw | before_TS | 60.72920 ± 0.42742 | 0.90846 ± 0.00125 | 10.22997 ± 0.51358 | 0.52730 ± 0.00146 |
| historical_test | KD_raw | after_TS | 60.72920 ± 0.42742 | 0.86139 ± 0.00612 | 3.43750 ± 0.38726 | 0.50942 ± 0.00386 |
| historical_test | KD_bias | before_TS | 60.63065 ± 0.15086 | 0.90777 ± 0.00068 | 9.16775 ± 0.12282 | 0.52941 ± 0.00022 |
| historical_test | KD_bias | after_TS | 60.63065 ± 0.15086 | 0.86770 ± 0.00189 | 3.41185 ± 0.20786 | 0.51363 ± 0.00112 |
| historical_test | KD_temperature | before_TS | 60.51657 ± 0.21663 | 0.87130 ± 0.00197 | 4.99030 ± 0.43127 | 0.51353 ± 0.00199 |
| historical_test | KD_temperature | after_TS | 60.51657 ± 0.21663 | 0.86162 ± 0.00391 | 3.04911 ± 0.37483 | 0.50947 ± 0.00277 |
| historical_test | KD_calibrated | before_TS | 59.86595 ± 1.06278 | 0.87067 ± 0.00307 | 5.26776 ± 0.67533 | 0.51347 ± 0.00299 |
| historical_test | KD_calibrated | after_TS | 59.86595 ± 1.06278 | 0.86182 ± 0.00658 | 3.15989 ± 0.54986 | 0.50968 ± 0.00452 |
| new_dev_holdout | KD_raw | before_TS | 63.50023 ± 1.20395 | 0.87018 ± 0.00412 | 9.61149 ± 0.59846 | 0.50239 ± 0.00055 |
| new_dev_holdout | KD_raw | after_TS | 63.50023 ± 1.20395 | 0.83642 ± 0.00315 | 5.94085 ± 0.82513 | 0.49261 ± 0.00238 |
| new_dev_holdout | KD_bias | before_TS | 64.76486 ± 0.29387 | 0.86654 ± 0.00163 | 10.59661 ± 1.25471 | 0.50330 ± 0.00061 |
| new_dev_holdout | KD_bias | after_TS | 64.76486 ± 0.29387 | 0.83998 ± 0.00173 | 6.68584 ± 0.59840 | 0.49523 ± 0.00075 |
| new_dev_holdout | KD_temperature | before_TS | 62.93161 ± 1.05172 | 0.83869 ± 0.00107 | 6.44952 ± 1.26278 | 0.49186 ± 0.00114 |
| new_dev_holdout | KD_temperature | after_TS | 62.93161 ± 1.05172 | 0.83369 ± 0.00213 | 5.58029 ± 2.03667 | 0.49092 ± 0.00151 |
| new_dev_holdout | KD_calibrated | before_TS | 62.10720 ± 1.89124 | 0.84976 ± 0.01769 | 6.28009 ± 1.85295 | 0.50046 ± 0.01381 |
| new_dev_holdout | KD_calibrated | after_TS | 62.10720 ± 1.89124 | 0.84443 ± 0.01878 | 6.27006 ± 1.21820 | 0.49890 ± 0.01345 |

## 核心比较：双方完成Student TS后，temperature-only − raw

| 集合 | 指标 | 差值 [95%配对区间] |
|---|---|---:|
| historical_test | macro_f1 | -0.212636 [-0.776793, +0.344248] |
| historical_test | nll | +0.000222 [-0.002437, +0.003008] |
| historical_test | ece10 | -0.388396 [-1.639927, +0.913370] |
| historical_test | brier | +0.000053 [-0.001486, +0.001592] |
| new_dev_holdout | macro_f1 | -0.568625 [-2.396979, +1.056183] |
| new_dev_holdout | nll | -0.002728 [-0.008267, +0.002709] |
| new_dev_holdout | ece10 | -0.360559 [-2.933718, +2.516721] |
| new_dev_holdout | brier | -0.001687 [-0.004944, +0.001501] |

F1/ECE差值为百分点。NLL、ECE、Brier负差值有利于前者。区间跨0不证明等效。
完整CSV包括：TS前/后各6种四格对比，以及每个方法自身TS前后差值；交互直接由联合抽样差分计算。
24个模型×集合组合的argmax与所有分类指标已检查完全不变。400条拟合NLL仅作优化诊断，不是独立评估成绩。

结果已保存： /content/drive/MyDrive/CLIP_Jev_Component_StudentTS_20261030T025735Z_48932a_results.zip


## 11. 可选下载
将最终 `_results.zip` 用于复核和论文更新。本任务不包括新评估集、base-teacher新训练或部署计时。

In [11]:
try:
    from google.colab import files
    files.download(str(RESULT_ZIP))
except ImportError:
    print(RESULT_ZIP)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>